# F#: leaving C# behind

## Same runtime, different language
F# is .NET. Same CLR, same `decimal`, same NuGet, same Visual Studio,
same `dotnet test`. You can reference an F# project from a C# project
and call into it like any other library. From the build system's
perspective, nothing exotic is happening.

But the **language** is different. F# is what you get when ML-family
people are allowed to ship on the CLR. Native discriminated unions.
Exhaustive `match` that the compiler will flag for you — and that you
can turn into a hard error with one project setting. Immutability by
default; you opt **in** to mutability, not out. F#'s own types don't
have `null`.

The idiomatic C# from the previous chapter closed six of the seven
footguns. It did so by **simulating** features the language doesn't have:
sealed-record DUs with a private constructor, `IReadOnlyList<T>` instead
of `List<T>`, switch expressions with a `_ => throw` arm so the
compiler stays quiet. Each simulation works. Each one leaves a smell.

From up here, those smells look like what they are: workarounds for
features F# ships natively.

## The same type, two languages

`RowDecision` is the centerpiece of both pipelines. Here it is in
idiomatic C#:

In [ ]:
// csharp-fuel-engine/src/FuelUploadEngine/Domain/RowDecision.cs
public abstract record RowDecision
{
    private RowDecision() { }

    public sealed record AcceptedTransaction(FuelTransaction Transaction) : RowDecision;

    public sealed record AcceptedTransactionWithWarnings(
        FuelTransaction Transaction,
        IReadOnlyList<UploadWarning> Warnings) : RowDecision;

    public sealed record QuarantinedRow : RowDecision
    {
        public QuarantinedRow(
            RowNumber rowNumber,
            FuelTransaction transaction,
            IReadOnlyList<QuarantineReason> reasons,
            IReadOnlyList<UploadWarning> warnings)
        {
            if (reasons.Count == 0)
                throw new ArgumentException("Quarantined rows require at least one reason.", nameof(reasons));
            // ... assign properties ...
        }
        // ... property declarations ...
    }

    public sealed record SkippedDuplicate(RowNumber RowNumber, DuplicateState Duplicate, DuplicateSkipCode Reason) : RowDecision;
    public sealed record RejectedRow(RowNumber RowNumber, RejectionReason Reason) : RowDecision;
    public sealed record FatalProcessingError(RowNumber RowNumber, FatalError Error) : RowDecision;
}

Roughly forty lines once you include the constructor invariant on
`QuarantinedRow`. Now the same type in F#:

In [ ]:
// fsharp-fuel-engine/FuelUpload.Domain/Decision.fs
[<RequireQualifiedAccess>]
type RowDecision =
    | Accepted of AcceptedTransaction
    | AcceptedWithWarnings of AcceptedTransaction * Warning list
    | Quarantined of QuarantinedRow
    | SkippedDuplicate of SkippedDuplicate
    | Rejected of RejectedRow
    | Fatal of FatalProcessingError

**Seven lines, no ceremony.** No `private RowDecision()` to lock the
hierarchy closed. No `sealed record` per case. No hand-written
constructor invariant — the "quarantine has at least one reason"
invariant is enforced one file over by a private constructor on a
`QuarantineReasons` wrapper type, applied once instead of per-case.

The consumer side is symmetric. Idiomatic C# uses a switch expression
that's *almost* exhaustive but has to close itself with a default arm:

In [ ]:
return duplicateCheck switch
{
    DuplicateCheckResult.NotDuplicate notDuplicate => CreateAcceptedDecision(...),
    DuplicateCheckResult.Duplicate duplicate      => DuplicatePolicy.Classify(...) ?? CreateAcceptedDecision(...),
    _ => throw new InvalidOperationException("Unhandled duplicate check result.")
};

That `_ => throw` is the price of doing DUs in a language that
doesn't have them. The compiler will check the *declared* arms for
type-correctness, but once you write `_ => ...` it stops asking about
new cases. F# has no equivalent need:

In [ ]:
match Validation.validateConfig config, vehicleLookup, duplicateCheck with
| fatal :: _, _, _                              -> RowDecision.Fatal fatal
| _, VehicleLookupResult.Fatal fatal, _         -> RowDecision.Fatal fatal
| _, _, DuplicateCheckResult.Fatal fatal        -> RowDecision.Fatal fatal
| [], _, _                                      -> ...

That's a pattern match on a **tuple of three sum types simultaneously**,
and the F# compiler will warn if any combination is unreachable or
missing. We'll come back to the "missing" case in a minute.

## The F# domain layout

Six files, compiled in this order. F# has no forward references; the
file order in the `.fsproj` is the dependency graph.

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 821.59375 990" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381578769_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381578769_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381578769_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381578769_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381578769_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381578769_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M281.211,91.241L254.311,98.534C227.411,105.827,173.612,120.414,146.712,135.207C119.813,150,119.813,165,119.813,172.5L119.813,180" id="L_P_FR_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_P_FR_0" data-points="W3sieCI6MjgxLjIxMDkzNzUsInkiOjkxLjI0MDY5MTE5Mjg2NTExfSx7IngiOjExOS44MTI1LCJ5IjoxMzV9LHsieCI6MTE5LjgxMjUsInkiOjE4NH1d" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M400.125,110L400.125,114.167C400.125,118.333,400.125,126.667,400.125,136.333C400.125,146,400.125,157,400.125,162.5L400.125,168" id="L_P_V_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_P_V_0" data-points="W3sieCI6NDAwLjEyNSwieSI6MTEwfSx7IngiOjQwMC4xMjUsInkiOjEzNX0seyJ4Ijo0MDAuMTI1LCJ5IjoxNzJ9XQ==" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M519.039,90.058L547.717,97.549C576.396,105.039,633.753,120.019,662.431,131.01C691.109,142,691.109,149,691.109,152.5L691.109,156" id="L_P_D_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_P_D_0" data-points="W3sieCI6NTE5LjAzOTA2MjUsInkiOjkwLjA1ODI2MTI4OTgwMjkzfSx7IngiOjY5MS4xMDkzNzUsInkiOjEzNX0seyJ4Ijo2OTEuMTA5Mzc1LCJ5IjoxNjB9XQ==" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M119.813,262L119.813,270.167C119.813,278.333,119.813,294.667,144.265,314.697C168.717,334.727,217.622,358.454,242.074,370.318L266.526,382.181" id="L_FR_Dec_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_FR_Dec_0" data-points="W3sieCI6MTE5LjgxMjUsInkiOjI2Mn0seyJ4IjoxMTkuODEyNSwieSI6MzExfSx7IngiOjI3MC4xMjUsInkiOjM4My45Mjc1MzYyMzE4ODQwNn1d" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M400.125,274L400.125,280.167C400.125,286.333,400.125,298.667,400.125,308.333C400.125,318,400.125,325,400.125,328.5L400.125,332" id="L_V_Dec_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_V_Dec_0" data-points="W3sieCI6NDAwLjEyNSwieSI6Mjc0fSx7IngiOjQwMC4xMjUsInkiOjMxMX0seyJ4Ijo0MDAuMTI1LCJ5IjozMzZ9XQ==" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M691.109,286L691.109,290.167C691.109,294.333,691.109,302.667,664.883,319.091C638.656,335.516,586.202,360.031,559.976,372.289L533.749,384.547" id="L_D_Dec_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_D_Dec_0" data-points="W3sieCI6NjkxLjEwOTM3NSwieSI6Mjg2fSx7IngiOjY5MS4xMDkzNzUsInkiOjMxMX0seyJ4Ijo1MzAuMTI1LCJ5IjozODYuMjQwNzIzODM2MTE2Nn1d" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M277.973,558L273.387,562.167C268.802,566.333,259.632,574.667,255.046,582.333C250.461,590,250.461,597,250.461,600.5L250.461,604" id="L_Dec_Val_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Dec_Val_0" data-points="W3sieCI6Mjc3Ljk3MjcxMzY5NDg1MjksInkiOjU1OH0seyJ4IjoyNTAuNDYwOTM3NSwieSI6NTgzfSx7IngiOjI1MC40NjA5Mzc1LCJ5Ijo2MDh9XQ==" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M530.125,557.268L535.181,561.557C540.237,565.846,550.349,574.423,555.405,592.211C560.461,610,560.461,637,560.461,650.5L560.461,664" id="L_Dec_BS_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Dec_BS_0" data-points="W3sieCI6NTMwLjEyNSwieSI6NTU3LjI2ODQ3OTI2NzE2Mzd9LHsieCI6NTYwLjQ2MDkzNzUsInkiOjU4M30seyJ4Ijo1NjAuNDYwOTM3NSwieSI6NjY4fV0=" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M250.461,830L250.461,834.167C250.461,838.333,250.461,846.667,259.137,855.239C267.814,863.812,285.167,872.624,293.843,877.03L302.519,881.436" id="L_Val_DE_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Val_DE_0" data-points="W3sieCI6MjUwLjQ2MDkzNzUsInkiOjgzMH0seyJ4IjoyNTAuNDYwOTM3NSwieSI6ODU1fSx7IngiOjMwNi4wODU5Mzc1LCJ5Ijo4ODMuMjQ2NTkzOTM0MzMyfV0=" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path><path d="M560.461,770L560.461,784.167C560.461,798.333,560.461,826.667,550.014,845.785C539.567,864.904,518.673,874.808,508.226,879.76L497.779,884.712" id="L_BS_DE_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_BS_DE_0" data-points="W3sieCI6NTYwLjQ2MDkzNzUsInkiOjc3MH0seyJ4Ijo1NjAuNDYwOTM3NSwieSI6ODU1fSx7IngiOjQ5NC4xNjQwNjI1LCJ5Ijo4ODYuNDI1MDM1MzI2MjE5NH1d" marker-end="url(#mermaid-1779381578769_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_P_FR_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_P_V_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_P_D_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_FR_Dec_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_V_Dec_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_D_Dec_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Dec_Val_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Dec_BS_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Val_DE_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_BS_DE_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-P-0" transform="translate(400.125, 59)"><rect class="basic label-container" style="" x="-118.9140625" y="-51" width="237.828125" height="102"></rect><g class="label" style="" transform="translate(-88.9140625, -36)"><rect></rect><foreignobject width="177.828125" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Primitives.fs<br>UploadMode DU<br>FatalProcessingError DU
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-FR-1" transform="translate(119.8125, 223)"><rect class="basic label-container" style="" x="-111.8125" y="-39" width="223.625" height="78"></rect><g class="label" style="" transform="translate(-81.8125, -24)"><rect></rect><foreignobject width="163.625" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
FuelRow.fs<br>ParsedFuelRow record
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-V-2" transform="translate(400.125, 223)"><rect class="basic label-container" style="" x="-118.5" y="-51" width="237" height="102"></rect><g class="label" style="" transform="translate(-88.5, -36)"><rect></rect><foreignobject width="177" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Vehicle.fs<br>Vehicle record<br>VehicleLookupResult DU
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-D-3" transform="translate(691.109375, 223)"><rect class="basic label-container" style="" x="-122.484375" y="-63" width="244.96875" height="126"></rect><g class="label" style="" transform="translate(-92.484375, -48)"><rect></rect><foreignobject width="184.96875" height="96">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Duplicate.fs<br>PreviousAttemptState DU<br>DuplicateCheckResult DU<br>DuplicateSkipReason DU
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Dec-4" transform="translate(400.125, 447)"><rect class="basic label-container" style="fill:#e6f5ff !important;stroke:#0066cc !important;stroke-width:2px !important" x="-130" y="-111" width="260" height="222"></rect><g class="label" style="" transform="translate(-100, -96)"><rect></rect><foreignobject width="200" height="192">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Decision.fs<br>AcceptedTransaction<br>RejectedRow / SkippedDuplicate<br>QuarantinedRow<br>RowDecision DU<br>FuelRowContext / ClassifiedRow
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Val-5" transform="translate(250.4609375, 719)"><rect class="basic label-container" style="" x="-130" y="-111" width="260" height="222"></rect><g class="label" style="" transform="translate(-100, -96)"><rect></rect><foreignobject width="200" height="192">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Validation.fs<br>ValidationConfig<br>ValidationError / Warning / QuarantineReason DUs<br>QuarantineReasons NonEmpty wrapper<br>pure validate / warnings / quarantine functions
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-BS-6" transform="translate(560.4609375, 719)"><rect class="basic label-container" style="" x="-130" y="-51" width="260" height="102"></rect><g class="label" style="" transform="translate(-100, -36)"><rect></rect><foreignobject width="200" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
BatchSummary.fs<br>summarize: ClassifiedRow list -&gt; BatchSummary
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-DE-7" transform="translate(400.125, 931)"><rect class="basic label-container" style="fill:#e6f5ff !important;stroke:#0066cc !important;stroke-width:2px !important" x="-94.0390625" y="-51" width="188.078125" height="102"></rect><g class="label" style="" transform="translate(-64.0390625, -36)"><rect></rect><foreignobject width="128.078125" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
DecisionEngine.fs<br>classifyRow<br>classifyBatch
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

The whole pure domain is around 400 lines. Idiomatic C# is closer to a
thousand for the same problem, because every DU costs roughly thirty
lines of encoding ceremony and every collection needs `IReadOnlyList<T>`
wrappers written explicitly.

## The recovery matrix

This is the climax. The fuel domain has four `UploadMode` cases and six
`PreviousAttemptState` cases, which means twenty-four `(mode, state)`
combinations, plus the "no duplicate" shortcut. The interesting subset —
what each recovery mode is allowed to accept — is a **matrix**.

Idiomatic C# writes it as a sequence of `if` statements on
`PreviousUploadOutcome`-like fields plus a delegate to
`DuplicatePolicy.ClassifyDuplicate`:

In [ ]:
// csharp-fuel-engine/src/FuelUploadEngine/Engine/FuelUploadDecisionEngine.cs
return duplicateCheck switch
{
    DuplicateCheckResult.NotDuplicate notDuplicate => CreateAcceptedDecision(...),
    DuplicateCheckResult.Duplicate duplicate => DuplicatePolicy.ClassifyDuplicate(
            row, vehicle, duplicate.State, validationConfig, mode)
        ?? CreateAcceptedDecision(row, vehicle, duplicate.State.ExistingTransactionKey, validationConfig),
    _ => throw new InvalidOperationException("Unhandled duplicate check result.")
};

The actual matrix is hidden inside `DuplicatePolicy.ClassifyDuplicate` —
a separate file with its own `switch` on `(mode, state)`, returning
`RowDecision?` and using `null` to mean "fall through to accepted."

F# inlines the entire matrix in `DecisionEngine.fs`. Quoted verbatim:

In [ ]:
// fsharp-fuel-engine/FuelUpload.Domain/DecisionEngine.fs
match mode, duplicateCheck with
| _, DuplicateCheckResult.NoDuplicate ->
    accepted config mode row vehicle
| UploadMode.Normal, DuplicateCheckResult.Duplicate previous ->
    skipped row mode previous DuplicateSkipReason.NormalModeDuplicate
| UploadMode.Retry, DuplicateCheckResult.Duplicate PreviousAttemptState.RetryableFailure ->
    accepted config mode row vehicle
| UploadMode.Retry, DuplicateCheckResult.Duplicate PreviousAttemptState.Finalized ->
    skipped row mode PreviousAttemptState.Finalized
        DuplicateSkipReason.RetryModeDuplicateAlreadyFinalized
| UploadMode.Retry, DuplicateCheckResult.Duplicate previous ->
    skipped row mode previous (DuplicateSkipReason.RetryModeDuplicateNotRetryable previous)
| UploadMode.ConservativeRecovery,
  DuplicateCheckResult.Duplicate PreviousAttemptState.FailedBeforeCanonicalFinalization ->
    accepted config mode row vehicle
| UploadMode.ConservativeRecovery, DuplicateCheckResult.Duplicate previous ->
    skipped row mode previous
        (DuplicateSkipReason.RecoveryModeDuplicateAlreadyCanonicalized previous)
| UploadMode.AggressiveRecovery,
  DuplicateCheckResult.Duplicate PreviousAttemptState.FailedBeforeCanonicalFinalization ->
    accepted config mode row vehicle
| UploadMode.AggressiveRecovery,
  DuplicateCheckResult.Duplicate
      PreviousAttemptState.FailedAfterCanonicalizationWithoutCanonicalTransactionKey ->
    accepted config mode row vehicle
| UploadMode.AggressiveRecovery, DuplicateCheckResult.Duplicate previous ->
    skipped row mode previous
        (DuplicateSkipReason.RecoveryModeDuplicateAlreadyCanonicalized previous)
| _, DuplicateCheckResult.Fatal fatal ->
    RowDecision.Fatal fatal

Every accepted-by-recovery case is a **typed pattern**, not a boolean
check on a flag. There are no `if (previous.WasCanonicalized)` branches,
no `?? CreateAccepted`, no nullable return. The cases that aren't
listed (e.g. `Retry` + `Finalized` is listed, but `Retry` +
`FailedAfterCanonicalizationWithCanonicalTransactionKey` is **not**
listed by name) fall through to the catch-all `Duplicate previous`
pattern earlier in the same arm — which the compiler can see is total
because every `(mode, state)` pair has *some* match.

If you compare to the C# `DuplicatePolicy.ClassifyDuplicate(...) ?? CreateAcceptedDecision(...)`
expression: the `??` is doing the work that pattern fall-through does
in F#, and that single `??` is the only thing standing between a row
that should have been skipped and a silent acceptance, if the policy
table forgets a case.

In F#, forgetting a case is a compiler warning at this exact site.

## Adding a 7th `RowDecision` case

Suppose product comes back next quarter: there's a seventh outcome,
`HeldForReview` — the row passed validation, isn't a duplicate, but
matches a fraud signal that needs human eyes before the canonical write.

**In F#.** Step 1, add the case:

In [ ]:
type RowDecision =
    | Accepted of AcceptedTransaction
    | AcceptedWithWarnings of AcceptedTransaction * Warning list
    | Quarantined of QuarantinedRow
    | SkippedDuplicate of SkippedDuplicate
    | Rejected of RejectedRow
    | Fatal of FatalProcessingError
    | HeldForReview of AcceptedTransaction * HeldReason   // <- one new line

Step 2, compile. Every `match` expression on `RowDecision` anywhere in
the solution now reports:

> **FS0025: Incomplete pattern matches on this expression.**
> For example, the value `HeldForReview (_, _)` may indicate a case not
> covered by the pattern(s).

The compiler tells you exactly which sites. `BatchSummary.summarize`
will flag. `Interop.toDecisionDto` will flag. Any consumer in the
solution that pattern-matches on `RowDecision` will flag. The warning
resolves only when every site handles the case.

With `<TreatWarningsAsErrors>true</TreatWarningsAsErrors>` (or the
narrower `<WarningsAsErrors>25</WarningsAsErrors>`) in the `.fsproj`,
it's a **hard build break**.

**In idiomatic C#.** Step 1, add the case:

In [ ]:
public sealed record HeldForReview(FuelTransaction Transaction, HeldReason Reason) : RowDecision;

Step 2, compile. **Nothing happens.** The switch expressions in
`FuelUploadDecisionEngine`, `BatchSummaryCalculator`, the response
mapper — all have a `_ => throw new InvalidOperationException(...)`
arm. They all still compile. They all still pass their existing tests.

At runtime, the first `HeldForReview` decision hits the default arm
and throws. You discover this in production, on a row whose payment
team is now waiting on a fraud team that doesn't know it has work.

This is **Footgun 3 ported into idiomatic C#**. The encoding is
cleaner. The underlying gap remains: C# can't enforce exhaustive
matching across an open hierarchy, and the `_ =>` arm makes every
closed hierarchy effectively open.

| Cost to add a case          | F#                          | Idiomatic C#                          |
| --------------------------- | --------------------------- | ------------------------------------- |
| Lines to modify the DU      | 1                           | 1                                     |
| Compiler help finding sites | yes (FS0025 at every site)  | no (`_ =>` arms hide them)            |
| Failure mode if you miss one| warning/error at compile    | `InvalidOperationException` at runtime |

## Easier or harder to read?

Honest answer: **both**. Less code, but the code is denser. The
concepts are unfamiliar.

A junior reading this snippet for the first time sees four ideas
stacked into four lines:

In [ ]:
match Validation.validateConfig config, vehicleLookup, duplicateCheck with
| fatal :: _, _, _                       -> RowDecision.Fatal fatal
| _, VehicleLookupResult.Fatal fatal, _  -> RowDecision.Fatal fatal
| _, _, DuplicateCheckResult.Fatal fatal -> RowDecision.Fatal fatal
| [], _, _                               -> ...

That's tuple construction, list deconstruction (`fatal :: _` means
"non-empty list, take the head"), wildcard patterns, and DU case
patterns. A first-time reader has to learn all four to parse the
snippet at all.

The equivalent idiomatic C# is more verbose but uses concepts every
C# dev already knows:

In [ ]:
if (vehicleLookup is VehicleLookupResult.Unavailable vehicleUnavailable)
    return new RowDecision.FatalProcessingError(row.RowNumber, vehicleUnavailable.Error);

if (duplicateCheck is DuplicateCheckResult.Unavailable duplicateUnavailable)
    return new RowDecision.FatalProcessingError(row.RowNumber, duplicateUnavailable.Error);

`if`, `is`, `return`. Three keywords. Lower density — you read more
lines to learn less — but you don't need to learn anything new to
read it.

The honest verdict: **once you're past week one of F#, the F# code is
dramatically easier to read** because the structure of the code matches
the structure of the problem. Sum types in, sum types out, exhaustive
match in the middle. The whole recovery matrix fits on one screen and
each row of the matrix is a row of code.

But week one is a real cost. The `|>` pipe, the `::` cons, the `_` for
"don't care," the way `match` returns a value instead of falling out
the bottom, the `[<RequireQualifiedAccess>]` attribute — every one of
those is a new concept. There is no language equivalent of "you
already know this from C#." If your team is new to F#, the learning
curve is non-trivial and lasts roughly a quarter.

## What still leaks

F# isn't a free lunch. The same CLR that gives you `decimal` and
`DateTimeOffset` also gives you `NullReferenceException` at any
interop boundary. `List.head []` still throws `ArgumentException` at
runtime — partial functions exist in the F# standard library; the
convention is `List.tryHead`, but the convention is opt-in. And there
is an entire class of issue called `[<CLIMutable>]` that you can see
in `Decision.fs` already:

In [ ]:
[<CLIMutable>]
type AcceptedTransaction =
    { TransactionId: string
      SourceRowNumber: int
      Vehicle: Vehicle
      OccurredAt: DateTimeOffset
      ...
      Mode: UploadMode }

That attribute tells the compiler to *also* emit a parameterless
constructor and settable properties for the record, so C# callers and
JSON deserializers can construct it. The F# code itself never mutates
these values — but the type, at the CLR level, exposes a mutable
shape. It is a deliberately carved hole in the safety guarantee, and
it exists for one reason: F# rarely runs alone.

That hole — the boundary between the gorgeous F# core and the C# world
that has to consume it — is the next chapter.

[Next chapter →](06-fsharp-interop.ipynb)